In [108]:
import json
import numpy as np
import pandas as pd
import random
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

In [109]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


#Load Datasets

In [110]:

train = pd.read_parquet('/content/drive/MyDrive/features/context_features/train_context.parquet')
val = pd.read_parquet('/content/drive/MyDrive/features/context_features/val_context.parquet')
test = pd.read_parquet('/content/drive/MyDrive/features/context_features/test_context.parquet')

train_user_features = pd.read_parquet('/content/drive/MyDrive/features/user_features/train_user_features.parquet')
val_user_features   = pd.read_parquet('/content/drive/MyDrive/features/user_features/val_user_features.parquet')
test_user_features  = pd.read_parquet('/content/drive/MyDrive/features/user_features/test_user_features.parquet')

news_path = '/content/drive/MyDrive/features/news_features/news_features.parquet'
news_features = pd.read_parquet(news_path)

news_entities_path = '/content/drive/MyDrive/embeddings/news_entities.parquet'
news_entities = pd.read_parquet(news_entities_path)



In [111]:
print(train_user_features.head(10))

   impression_id  user_id                time  \
0              1        1 2019-11-11 09:05:58   
1              2        2 2019-11-12 18:11:30   
2              4        4 2019-11-11 05:28:05   
3              5        5 2019-11-12 16:11:21   
4              6        6 2019-11-11 18:52:13   
5              6        6 2019-11-11 18:52:13   
6              7        7 2019-11-11 12:22:09   
7              8        8 2019-11-12 22:29:36   
8             10       10 2019-11-11 11:28:11   
9             13       13 2019-11-11 05:52:04   

                                             history  pos_id  \
0  [26659, 7649, 10157, 13853, 11735, 5063, 22437...   17918   
1  [19437, 27124, 48998, 22350, 37334, 22912, 447...   37331   
2  [49779, 14969, 23815, 48261, 33954, 15429, 511...   43802   
3                        [26331, 5983, 16352, 33322]   20105   
4  [5143, 38345, 35829, 24011, 7704, 47885, 43992...    5498   
5  [5143, 38345, 35829, 24011, 7704, 47885, 43992...   39158   
6  [43393, 4

In [112]:
print(news_features.head(10))

   news_id                subclustered_category  bert_emb_0  bert_emb_1  \
0    26629                      lifestyleroyals   -0.006567    0.030011   
1    45241                           weightloss    0.013164    0.027189   
2    49591  CounterTerrorism_MiddleEastConflict   -0.018924    0.047626   
3    28765                       health_general    0.047500    0.066250   
4    28005                              medical    0.000916    0.073814   
5    31230                      NFL_StarPlayers    0.015787    0.040826   
6    43056                 ClimateChangeImpacts    0.010033   -0.063057   
7    47796         IntlHealthRisk_UnusualEvents    0.035753   -0.017440   
8    36210                               gaming   -0.020225   -0.045939   
9     6638             newsscienceandtechnology    0.032412   -0.025327   

   bert_emb_2  bert_emb_3  bert_emb_4  bert_emb_5  bert_emb_6  bert_emb_7  \
0    0.041711    0.008560    0.023631    0.013891    0.019353   -0.043506   
1    0.018828    0.0

In [113]:
#map entities news ID to integers
news_map_path = '/content/drive/MyDrive/data/global_id_mappings/global_news_id_map.json'
with open(news_map_path, 'r') as f:
    news_id_map = json.load(f)

news_entities['news_id'] = news_entities['news_id'].map(news_id_map)

In [114]:
print(news_entities.head(10))

   news_id                                       kg_title_emb  \
0    26629  [0.008122858, -0.07991525, -0.016764905, 0.158...   
1    45241  [-0.029497119, -0.021168852, 0.03713986, -0.11...   
2    49591  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...   
3    28765  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...   
4    28005  [0.045086294, 0.05803315, 0.016441302, 0.00560...   
5    31230  [0.06864393, -0.047789503, -0.076707, 0.045057...   
6    43056  [-0.1300684, -0.14033258, 0.03021715, -0.10082...   
7    47796  [0.01414296, -0.09023139, -0.0042716097, -0.07...   
8    36210  [0.116379745, -0.088518046, 0.067738116, -0.04...   
9     6638  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...   

                                     kg_abstract_emb  kg_title_flag  \
0  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...              1   
1  [-0.029497119, -0.021168852, 0.03713986, -0.11...              1   
2  [-0.06553931, -0.08845358, -0.015253108, -0.03...              0   


# CBF Model

In [115]:
def build_article_matrices(news_features: pd.DataFrame, news_entities: pd.DataFrame):
    common_ids    = sorted(set(news_features["news_id"]) & set(news_entities.index))
    nf_indexed    = news_features.set_index("news_id").loc[common_ids]
    ne_indexed    = news_entities.loc[common_ids]

    article_ids   = common_ids
    article_index = {nid: i for i, nid in enumerate(article_ids)}

    # tfidf matrix — topic_0 to topic_49
    topic_cols   = [f"topic_{i}" for i in range(50)]
    tfidf_matrix = nf_indexed[topic_cols].values.astype(np.float32)

    # entity matrix — stack KG + NER embeddings + flags
    kg_title_mat     = np.stack(ne_indexed["kg_title_emb"].values)
    kg_abstract_mat  = np.stack(ne_indexed["kg_abstract_emb"].values)
    ner_title_mat    = np.stack(ne_indexed["ner_title_emb"].values)
    ner_abstract_mat = np.stack(ne_indexed["ner_abstract_emb"].values)
    flags_mat        = ne_indexed[
        ["kg_title_flag", "kg_abstract_flag", "ner_title_flag", "ner_abstract_flag"]
    ].values.astype(np.float32)

    entity_matrix = np.concatenate([
        kg_title_mat, kg_abstract_mat,
        ner_title_mat, ner_abstract_mat,
        flags_mat
    ], axis=1).astype(np.float32)

    print(f"Article matrices built: {len(article_ids):,} articles")
    print(f"  TF-IDF matrix shape : {tfidf_matrix.shape}")
    print(f"  Entity matrix shape : {entity_matrix.shape}")
    return tfidf_matrix, entity_matrix, article_ids, article_index



In [116]:
def build_user_profiles(
    behaviors     : pd.DataFrame,
    tfidf_matrix  : np.ndarray,
    entity_matrix : np.ndarray,
    article_index : dict,
) -> tuple:
    tfidf_dim  = tfidf_matrix.shape[1]
    entity_dim = entity_matrix.shape[1]

    user_tfidf_profiles  = {}
    user_entity_profiles = {}

    user_histories = behaviors[["user_id", "history"]].drop_duplicates(subset="user_id")

    for _, row in user_histories.iterrows():
        user_id      = row["user_id"]
        history      = row["history"]
        history_list = []

        if history is not None:
            if isinstance(history, list):
                history_list = history
            elif hasattr(history, "__len__"):
                history_list = list(history)

        valid_idxs = [article_index[nid] for nid in history_list if nid in article_index]

        if not valid_idxs:
            user_tfidf_profiles[user_id]  = np.zeros(tfidf_dim,  dtype=np.float32)
            user_entity_profiles[user_id] = np.zeros(entity_dim, dtype=np.float32)
        else:
            user_tfidf_profiles[user_id]  = tfidf_matrix[valid_idxs].mean(axis=0)
            user_entity_profiles[user_id] = entity_matrix[valid_idxs].mean(axis=0)

    print(f"User profiles built for {len(user_tfidf_profiles):,} users.")
    return user_tfidf_profiles, user_entity_profiles



In [117]:
def cosine_sim(a: np.ndarray, b: np.ndarray) -> float:
    norm_a = np.linalg.norm(a)
    norm_b = np.linalg.norm(b)
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return float(np.dot(a, b) / (norm_a * norm_b))

In [118]:
def compute_features(
    user_id              : int,
    news_id              : int,
    recency              : float,
    user_tfidf_profiles  : dict,
    user_entity_profiles : dict,
    tfidf_matrix         : np.ndarray,
    entity_matrix        : np.ndarray,
    article_index        : dict,
    news_cat_map         : dict,
    user_top_cats        : dict,
) -> np.ndarray:

    article_idx = article_index.get(news_id, None)

    # semantic match
    user_tfidf = user_tfidf_profiles.get(user_id, None)
    if user_tfidf is not None and article_idx is not None:
        semantic_match = cosine_sim(user_tfidf, tfidf_matrix[article_idx])
    else:
        semantic_match = 0.0

    # knowledge match
    user_entity = user_entity_profiles.get(user_id, None)
    if user_entity is not None and article_idx is not None:
        knowledge_match = cosine_sim(user_entity, entity_matrix[article_idx])
    else:
        knowledge_match = 0.0

    # category match
    article_cat = news_cat_map.get(news_id, None)
    top_cats    = user_top_cats.get(user_id, ("cold_start", "cold_start", "cold_start"))
    if article_cat is None:
        cat_match = 0.0
    elif article_cat == top_cats[0]:
        cat_match = 1.0
    elif article_cat in top_cats[1:]:
        cat_match = 0.5
    else:
        cat_match = 0.0

    return np.array([semantic_match, knowledge_match, cat_match, recency], dtype=np.float32)

In [119]:
def build_train_features(
    train_context        : pd.DataFrame,
    user_tfidf_profiles  : dict,
    user_entity_profiles : dict,
    tfidf_matrix         : np.ndarray,
    entity_matrix        : np.ndarray,
    article_index        : dict,
    news_cat_map         : dict,
    user_top_cats        : dict,
) -> tuple:
    X, y = [], []

    for _, row in train_context.iterrows():
        user_id    = row["user_id"]
        candidates = list(row["candidate_articles"])
        rec_scores = list(row["recency_scores"])
        rec_map    = dict(zip(candidates, rec_scores))

        # positive article
        pos_id  = int(row["pos_id"])
        recency = float(rec_map.get(pos_id, 0.0))
        feat    = compute_features(
            user_id, pos_id, recency,
            user_tfidf_profiles, user_entity_profiles,
            tfidf_matrix, entity_matrix, article_index,
            news_cat_map, user_top_cats,
        )
        X.append(feat)
        y.append(1)

        # negative articles
        for neg_id in list(row["neg_ids"]):
            neg_id  = int(neg_id)
            recency = float(rec_map.get(neg_id, 0.0))
            feat    = compute_features(
                user_id, neg_id, recency,
                user_tfidf_profiles, user_entity_profiles,
                tfidf_matrix, entity_matrix, article_index,
                news_cat_map, user_top_cats,
            )
            X.append(feat)
            y.append(0)

    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int32)

In [120]:
def train_logistic_regression(X_train: np.ndarray, y_train: np.ndarray):
    scaler   = StandardScaler()
    X_scaled = scaler.fit_transform(X_train)

    model = LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=42,
    )
    model.fit(X_scaled, y_train)

    print("Logistic regression trained.")
    print("Feature weights [semantic, knowledge, category, recency]:")
    print(f"  {model.coef_[0].round(4)}")
    print(f"Intercept: {model.intercept_[0]:.4f}")
    return model, scaler

In [121]:
def compute_mrr(labels: list, scores: list) -> float:
    ranked = [l for _, l in sorted(zip(scores, labels), reverse=True)]
    for rank, label in enumerate(ranked, start=1):
        if label == 1:
            return 1.0 / rank
    return 0.0


def compute_ndcg(labels: list, scores: list, k: int) -> float:
    ranked = [l for _, l in sorted(zip(scores, labels), reverse=True)][:k]
    dcg    = sum((2**l - 1) / np.log2(r + 2) for r, l in enumerate(ranked))
    ideal  = sorted(labels, reverse=True)[:k]
    idcg   = sum((2**l - 1) / np.log2(r + 2) for r, l in enumerate(ideal))
    return dcg / idcg if idcg > 0 else 0.0


def compute_map(labels: list, scores: list) -> float:
    ranked        = [l for _, l in sorted(zip(scores, labels), reverse=True)]
    num_positives = sum(labels)
    if num_positives == 0:
        return 0.0
    hits, precisions = 0, []
    for rank, label in enumerate(ranked, start=1):
        if label == 1:
            hits += 1
            precisions.append(hits / rank)
    return sum(precisions) / num_positives

In [122]:
def evaluate(
    eval_df              : pd.DataFrame,
    model                : LogisticRegression,
    scaler               : StandardScaler,
    user_tfidf_profiles  : dict,
    user_entity_profiles : dict,
    tfidf_matrix         : np.ndarray,
    entity_matrix        : np.ndarray,
    article_index        : dict,
    news_cat_map         : dict,
    user_top_cats        : dict,
) -> dict:

    aucs, mrrs, ndcg5s, ndcg10s, maps = [], [], [], [], []
    skipped = 0

    for _, row in eval_df.iterrows():

        user_id        = row["user_id"]
        all_candidates = list(row["candidate_articles"])
        labels         = list(row["labels"])
        rec_scores     = list(row["recency_scores"])
        recency_lookup = dict(zip(all_candidates, rec_scores))

        if sum(labels) == 0:
            skipped += 1
            continue

        feats = np.array([
            compute_features(
                user_id, int(nid),
                float(recency_lookup.get(nid, 0.0)),
                user_tfidf_profiles, user_entity_profiles,
                tfidf_matrix, entity_matrix, article_index,
                news_cat_map, user_top_cats,
            )
            for nid in all_candidates
        ], dtype=np.float32)

        feats_scaled = scaler.transform(feats)
        scores       = model.predict_proba(feats_scaled)[:, 1].tolist()

        if len(set(labels)) > 1:
            aucs.append(roc_auc_score(labels, scores))
        mrrs.append(compute_mrr(labels, scores))
        ndcg5s.append(compute_ndcg(labels, scores, k=5))
        ndcg10s.append(compute_ndcg(labels, scores, k=10))
        maps.append(compute_map(labels, scores))

    print(f"Skipped {skipped:,} impressions (no positives)")
    return {
        "AUC"    : float(np.mean(aucs))    if aucs    else 0.0,
        "MRR"    : float(np.mean(mrrs))    if mrrs    else 0.0,
        "MAP"    : float(np.mean(maps))    if maps    else 0.0,
        "nDCG@5" : float(np.mean(ndcg5s)) if ndcg5s  else 0.0,
        "nDCG@10": float(np.mean(ndcg10s))if ndcg10s else 0.0,
    }

## Run CBF model

In [123]:
# build article content matrices
tfidf_matrix, entity_matrix, article_ids, article_index = build_article_matrices(
    news_features, news_entities
)

Article matrices built: 51,281 articles
  TF-IDF matrix shape : (51281, 50)
  Entity matrix shape : (51281, 972)


In [124]:
# build user profiles from all splits combined
# train users get history from train, val/test users from their own rows
# this ensures cold-start val/test users still get a profile from their history
all_behaviors = pd.concat([
    train_user_features[["user_id", "history"]],
    val_user_features[["user_id", "history"]],
    test_user_features[["user_id", "history"]]
], ignore_index=True).drop_duplicates(subset="user_id")

user_tfidf_profiles, user_entity_profiles = build_user_profiles(
    all_behaviors, tfidf_matrix, entity_matrix, article_index
)

User profiles built for 50,000 users.


In [125]:
# build lookup maps
news_cat_map = dict(zip(news_features["news_id"], news_features["subclustered_category"]))

train_user_dedup = train_user_features.drop_duplicates(subset="user_id", keep="last")
user_top_cats = {
    row["user_id"]: (row["top_cat_1"], row["top_cat_2"], row["top_cat_3"])
    for _, row in train_user_dedup.iterrows()
}

for df in [val_user_features, test_user_features]:
    dedup = df.drop_duplicates(subset="user_id", keep="last")
    for _, row in dedup.iterrows():
        if row["user_id"] not in user_top_cats:
            user_top_cats[row["user_id"]] = (
                row["top_cat_1"], row["top_cat_2"], row["top_cat_3"]
            )

print(f"user_top_cats built for {len(user_top_cats):,} users")

user_top_cats built for 50,000 users


In [126]:
# build training feature matrix
print("\nBuilding training feature matrix...")
X_train, y_train = build_train_features(
    train,
    user_tfidf_profiles, user_entity_profiles,
    tfidf_matrix, entity_matrix, article_index,
    news_cat_map, user_top_cats,
)


Building training feature matrix...


In [127]:
#train logistic regression
model, scaler = train_logistic_regression(X_train, y_train)

Logistic regression trained.
Feature weights [semantic, knowledge, category, recency]:
  [ 0.1886 -0.0375  0.2274  0.0695]
Intercept: -0.0300


In [128]:
#evaluate on val
print("\nValidation results:")
val_results = evaluate(
    val, model, scaler,
    user_tfidf_profiles, user_entity_profiles,
    tfidf_matrix, entity_matrix, article_index,
    news_cat_map, user_top_cats,
)
for metric, value in val_results.items():
    print(f"  {metric}: {value:.4f}")


Validation results:
Skipped 0 impressions (no positives)
  AUC: 0.5870
  MRR: 0.2921
  MAP: 0.2921
  nDCG@5: 0.2950
  nDCG@10: 0.3464


In [129]:
# evaluate on test
print("\nTest results:")
test_results = evaluate(
    test, model, scaler,
    user_tfidf_profiles, user_entity_profiles,
    tfidf_matrix, entity_matrix, article_index,
    news_cat_map, user_top_cats,
)
for metric, value in test_results.items():
    print(f"  {metric}: {value:.4f}")



Test results:
Skipped 0 impressions (no positives)
  AUC: 0.5782
  MRR: 0.2825
  MAP: 0.2825
  nDCG@5: 0.2836
  nDCG@10: 0.3373


In [130]:
# build val feature matrix
X_val, y_val = [], []

for _, row in val.iterrows():
    user_id        = row["user_id"]
    all_candidates = list(row["candidate_articles"])
    labels         = list(row["labels"])
    rec_scores     = list(row["recency_scores"])
    recency_lookup = dict(zip(all_candidates, rec_scores))

    for nid, label in zip(all_candidates, labels):
        feat = compute_features(
            user_id, int(nid),
            float(recency_lookup.get(nid, 0.0)),
            user_tfidf_profiles, user_entity_profiles,
            tfidf_matrix, entity_matrix, article_index,
            news_cat_map, user_top_cats,
        )
        X_val.append(feat)
        y_val.append(label)

X_val      = np.array(X_val, dtype=np.float32)
y_val      = np.array(y_val, dtype=np.int32)
X_val_scaled = scaler.transform(X_val)

print(f"Val matrix: {X_val.shape}")

Val matrix: (1215336, 4)


In [131]:
#ablation test
feature_names = ["semantic_match", "knowledge_match", "cat_match", "recency"]

# baseline — all 4 features
baseline_auc = roc_auc_score(y_val, model.predict_proba(X_val_scaled)[:, 1])
print(f"Baseline (all 4 features): val AUC = {baseline_auc:.4f}\n")

# drop one feature at a time
for drop_idx, drop_name in enumerate(feature_names):
    keep_idxs     = [i for i in range(4) if i != drop_idx]
    X_train_ab    = X_train[:, keep_idxs]
    X_val_ab      = X_val[:, keep_idxs]

    scaler_ab     = StandardScaler().fit(X_train_ab)
    model_ab      = LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=42
    ).fit(scaler_ab.transform(X_train_ab), y_train)

    val_preds_ab  = model_ab.predict_proba(scaler_ab.transform(X_val_ab))[:, 1]
    auc_ab        = roc_auc_score(y_val, val_preds_ab)
    diff          = auc_ab - baseline_auc

    print(f"Drop {drop_name:20s}: val AUC = {auc_ab:.4f}  ({diff:+.4f})")

Baseline (all 4 features): val AUC = 0.6110

Drop semantic_match      : val AUC = 0.5958  (-0.0152)
Drop knowledge_match     : val AUC = 0.6100  (-0.0010)
Drop cat_match           : val AUC = 0.5958  (-0.0152)
Drop recency             : val AUC = 0.5830  (-0.0280)
